In [ ]:
# Practice Problems Day 17

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code: one transformer block written as a class, and a review classifier built around it. Plan on about 45 minutes. Each training run takes a few seconds on a laptop.

**Solutions** are in `HW-day17-solutions.ipynb` in this folder. Try every problem yourself first, then check your work.

**Reading:** Geron chapter 15, **Positional encodings**, **Multi-Head Attention (MHA)** and **Building the rest of the Transformer**.

GOALS:

1. Write a transformer block as a class: multi-head attention, skip, layer norm, feed-forward.
2. Use the block on the long reviews with one feeling word, where the RNN failed.
3. Explain in words what positions and the other pieces of the block are for.

**How much of this should you write yourself?** The data cell is given, with the training function from the notes. Problem 1 gives you most of the block, you write the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 17 practice"
git push
```

---

In [ ]:
# RUN THIS. The words, both review makers and the training function from the notes.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words
vocab_size = len(vocab)

# One feeling word reviews (like the RNN notes): ONE feeling word at a random spot decides
def make_reviews_one(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling
    y = (feeling < 5).long()
    return X, y

# Today's reviews: TWO feeling words, the LAST one decides
def make_reviews_two(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    positive = torch.randint(0, 5, (n_reviews,), generator=generator)
    negative = torch.randint(5, 10, (n_reviews,), generator=generator)
    spot_pos = torch.randint(0, n_words, (n_reviews,), generator=generator)
    spot_neg = (spot_pos + torch.randint(1, n_words, (n_reviews,), generator=generator)) % n_words
    X[torch.arange(n_reviews), spot_pos] = positive
    X[torch.arange(n_reviews), spot_neg] = negative
    y = (spot_pos > spot_neg).long()
    return X, y

def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))
    return valid_accuracy

print("vocabulary size:", vocab_size)

## Problem 1: Write the transformer block

**What to copy, what to write:** Most of the block is given. Write the lines the comments ask for. The notes have the same class, but try to write it without looking first. The picture of the block in the notes is a good guide.

**1a.** Fill in `forward` below. Then send 8 made up reviews of 10 words, `torch.randn(8, 10, 16)`, through a new block and print the shape that comes out.

**1b.** Count the numbers your block has to learn with `n_heads=2`. Then make a block with `n_heads=4` and count again. Did the count change? Use the picture of multi-head attention to explain why.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_numbers)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)

    def forward(self, X):
        # Send X into self.attention as the queries, keys and values. It gives back TWO things.

        # Add X back in (the skip), then self.norm1. Call the result Z.

        # Send Z through self.feed_forward, add Z back in, then self.norm2. Call the result out.

        return out

# Send 8 made up reviews of 10 words through a new block and print the shape that comes out
torch.manual_seed(42)
block = TransformerBlock()
X_fake = torch.randn(8, 10, 16)
# Send X_fake through block and print the shape. It should be 3 numbers: (reviews, words, numbers).


In [ ]:
# 1b: count the numbers to learn with 2 heads, then with 4 heads


*Your answers here.*

---

## Problem 2: The long reviews with one feeling word (steps outlined)

In the RNN notes a plain RNN got about 0.5 on 100 word reviews where **one** feeling word, anywhere in the review, decides. The GRU learned them after a few epochs. Now let's try your transformer block!

**What to copy, what to write:** Copy the `ReviewTransformer` class from the notes (it uses your `TransformerBlock`). Write the steps below yourself.

**2a.** Make the reviews with one feeling word, 100 words long, with `make_reviews_one` (4,000 to train on with `seed=0`, 1,000 to validate on with `seed=1`). Train a `ReviewTransformer` **without** positions on them for 5 epochs.

**2b.** How does it compare to the RNN (0.493 after 5 epochs)? Give two reasons it can learn these reviews: one about **positions** and one about how far the gradient has to travel back to the feeling word.

In [ ]:
# Step 1: copy the ReviewTransformer class from the notes

# Step 2: make the 100 word reviews with one feeling word with make_reviews_one (train with seed=0, validate with seed=1)

# Step 3: make a ReviewTransformer with n_words=100 and use_positions=False (set torch.manual_seed(42) first)

# Step 4: train it for 5 epochs with train_model
# It prints 5 lines, one validation accuracy per epoch.


*Your answers here.*

---

## Problem 3: In words

**3a.** "dog bites man" and "man bites dog" use the same three words. Explain to a friend in DATA 101 why a transformer block with no positions would give the same answer for both, and what the positional embedding adds to fix it.

**3b.** A friend says "a transformer reads the review one word at a time, like an RNN." What would you tell them? Use the words **attention** and **hidden state**.

*Your answers here.*

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Make the two feeling word reviews from the notes (`make_reviews_two`, 20 words, `seed=0` and `seed=1`) and train a `ReviewTransformer` with positions, but with `n_heads=4` in the block. Compare to the notes (0.956 after 10 epochs with 2 heads). What happens if you try `n_heads=3`?

In [ ]:
# your code here

*Your answers here.*